In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import RegexpTokenizer, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer

for package in ["punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "wordnet", "omw-1.4"]:
    nltk.download(package, quiet=True)

stop_words = set(stopwords.words("english"))

In [ ]:
raw_tickets = globals().get("tickets", globals().get("TICKETS", None))
if raw_tickets is None:
    raise NameError("Run the starter notebook first and set raw_tickets to its generated ticket list.")
raw_tickets = list(raw_tickets)

processed = [process_ticket(ticket) for ticket in raw_tickets]

def get_tokens(result, raw):
    if isinstance(result, dict):
        for key in ["final_tokens", "final", "tokens_final"]:
            if key in result:
                return list(result[key])
    if isinstance(result, (list, tuple)):
        for item in reversed(result):
            if isinstance(item, (list, tuple)) and all(isinstance(x, str) for x in item):
                return list(item)
    if isinstance(result, str):
        return result.split()
    return [x.lower() for x in word_tokenize(raw) if x.lower() not in stop_words and x not in ".,!?;:()[]{}"]

final_token_lists = [get_tokens(result, raw) for result, raw in zip(processed, raw_tickets)]
rows = []
for i, (raw, result, final_tokens) in enumerate(zip(raw_tickets, processed, final_token_lists), 1):
    tokens = word_tokenize(raw)
    punctuation = [t for t in tokens if re.fullmatch(r"[^\w\s]+", t)]
    sw = [t for t in tokens if t.lower() in stop_words]
    rows.append({
        "Ticket": i,
        "Sentences": len(nltk.sent_tokenize(raw)),
        "Tokens": len(tokens),
        "Punctuation tokens": len(punctuation),
        "Stopwords": len(sw),
        "Unique tokens": len(set(tokens)),
        "Final tokens": len(final_tokens)
    })

preprocessing_table = pd.DataFrame(rows)
count_cols = ["Sentences", "Tokens", "Punctuation tokens", "Stopwords", "Final tokens"]
total_row = {"Ticket": "Total"}
for col in count_cols:
    total_row[col] = preprocessing_table[col].sum()
total_row["Unique tokens"] = preprocessing_table["Unique tokens"].sum()
preprocessing_table = pd.concat([preprocessing_table, pd.DataFrame([total_row])], ignore_index=True)
display(preprocessing_table)

selected = 0
selected_tokens = word_tokenize(raw_tickets[selected])
selected_punctuation = [t for t in selected_tokens if re.fullmatch(r"[^\w\s]+", t)]
selected_stopwords = [t for t in selected_tokens if t.lower() in stop_words]
print("Selected ticket:", raw_tickets[selected])
print("Actual tokens:", selected_tokens)
print("Punctuation tokens:", selected_punctuation)
print("Stopwords:", selected_stopwords)
print("Final tokens:", final_token_lists[selected])

In [ ]:
patterns = {
    "Words longer than 5 letters": r"\b[A-Za-z]{6,}\b",
    "Numbers": r"(?<![\w.])\d+(?:\.\d+)+(?![\w.])|(?<![\w.])\d+(?![\w.])",
    "Capitalized words": r"\b[A-Z][a-zA-Z]*\b",
    "Words beginning with a vowel": r"\b[aeiouAEIOU][A-Za-z]*\b"
}

for i, ticket in enumerate(raw_tickets, 1):
    print(f"Ticket {i}")
    for name, pattern in patterns.items():
        print(name + ":", re.findall(pattern, ticket))

def replace_sensitive(text):
    text = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", "<EMAIL>", text)
    text = re.sub(r"https?://[^\s<>]+|www\.[^\s<>]+", "<URL>", text)
    text = re.sub(r"(?<!\w)(?:\+?\d[\d ().-]{7,}\d)(?!\w)", "<PHONE>", text)
    return text

for ticket in raw_tickets:
    print(replace_sensitive(ticket))

if "TEST_LINES" in globals():
    for line in TEST_LINES:
        print(line, "->", replace_sensitive(line))

In [ ]:
token_pattern = r"https?://[^\s<>]+|www\.[^\s<>]+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|\+?\d[\d ().-]{7,}\d|\d+(?:\.\d+)+|[A-Za-z]+(?:[-'][A-Za-z]+)*|\d+|[^\w\s]"
my_tokenizer = RegexpTokenizer(token_pattern)

def my_tokenize(text):
    return my_tokenizer.tokenize(text)

test_lines = globals().get("TEST_LINES", [])
print("TEST_LINES")
for line in test_lines:
    print("Text:", line)
    print("My tokenizer:", my_tokenize(line))
    print("word_tokenize:", word_tokenize(line))

print("Generated tickets")
for ticket in raw_tickets:
    print("Text:", ticket)
    print("My tokenizer:", my_tokenize(ticket))
    print("word_tokenize:", word_tokenize(ticket))

In [ ]:
porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp_stemmer = RegexpStemmer(r"(ing|ed|s)$", min=4)
lemmatizer = WordNetLemmatizer()

unique_final_tokens = sorted(set(token.lower() for group in final_token_lists for token in group if token.isalpha()))

def penn_to_wordnet(tag):
    first = tag[0].upper()
    if first == "J":
        return "a"
    if first == "V":
        return "v"
    if first == "N":
        return "n"
    if first == "R":
        return "r"
    return "n"

tagged = nltk.pos_tag(unique_final_tokens)
pos_map = {word: penn_to_wordnet(tag) for word, tag in tagged}

stem_rows = []
for word in unique_final_tokens:
    outputs = {
        "Word": word,
        "Porter": porter.stem(word),
        "Lancaster": lancaster.stem(word),
        "Regexp": regexp_stemmer.stem(word),
        "Lemma": lemmatizer.lemmatize(word, pos_map.get(word, "n"))
    }
    if len(set(outputs.values())) > 1:
        stem_rows.append(outputs)

stem_table = pd.DataFrame(stem_rows, columns=["Word", "Porter", "Lancaster", "Regexp", "Lemma"])
display(stem_table)

print("Regexp suffix choice: ing, ed and s were selected because words with these endings can occur in the ticket dataset, for example words such as 'loading', 'failed' and 'updates' if they appear in the generated tickets.")

In [ ]:
all_tokens = [token.lower() for ticket in raw_tickets for token in word_tokenize(ticket)]
s1 = all_tokens
s2 = [token for token in s1 if not re.fullmatch(r"[^\w\s]+", token)]
s3 = [token.lower() for group in final_token_lists for token in group]
s4 = [porter.stem(token) for token in s3]

def corpus_stats(name, tokens):
    return {
        "Set": name,
        "Total tokens": len(tokens),
        "Unique tokens": len(set(tokens)),
        "TTR": len(set(tokens)) / len(tokens) if tokens else 0
    }

stats_table = pd.DataFrame([
    corpus_stats("S1", s1),
    corpus_stats("S2", s2),
    corpus_stats("S3", s3),
    corpus_stats("S4", s4)
])
display(stats_table)

print("Top 10 words of S1")
display(pd.Series(s1).value_counts().head(10).rename_axis("Word").reset_index(name="Frequency"))

print("Top 10 words of S3")
display(pd.Series(s3).value_counts().head(10).rename_axis("Word").reset_index(name="Frequency"))

frequencies = pd.Series(s3).value_counts()
ranked = pd.DataFrame({"Rank": range(1, len(frequencies) + 1), "Frequency": frequencies.values})
ranked = ranked[(ranked["Rank"] > 0) & (ranked["Frequency"] > 0)]
plt.figure(figsize=(8, 5))
plt.plot(ranked["Rank"], ranked["Frequency"], marker=".")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("log(Rank)")
plt.ylabel("log(Frequency)")
plt.title("Log Rank vs Log Frequency for S3")
plt.grid(True)
plt.show()

def top_bigrams(tokens):
    bigrams = list(nltk.ngrams(tokens, 2))
    return pd.Series(bigrams).value_counts().head(5).rename_axis("Bigram").reset_index(name="Frequency")

print("Top 5 bigrams of S1")
display(top_bigrams(s1))
print("Top 5 bigrams of S3")
display(top_bigrams(s3))